# KNN Classification

K-Nearest Neighbors is one of the most straightforward algorithms in machine learning. It stores every training example and classifies new points by a majority vote among their K closest neighbors. This notebook covers the algorithm's intuition, key hyperparameters, distance metrics, and practical implementation with scikit-learn.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_classification, make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

np.random.seed(42)

---
## 1. The KNN Algorithm: Intuition

KNN is a **lazy learner** — it performs no computation during training and instead memorizes the entire dataset. When a new sample arrives, the algorithm:

1. Computes the distance from the new sample to every training sample.
2. Selects the **K** closest training samples.
3. Assigns the class that appears most frequently among those K neighbors (majority vote).

Because KNN makes no assumptions about the underlying data distribution, it is called a **non-parametric** method.

In [ ]:
# Simple visual: a new point among labeled neighbors
fig, ax = plt.subplots(figsize=(6, 5))

# Two classes of training points
class_a = np.array([[1, 2], [1.5, 1.8], [2, 2.5], [1.2, 3]])
class_b = np.array([[3, 3], [3.5, 2.5], [4, 3.5], [3.8, 2]])

ax.scatter(class_a[:, 0], class_a[:, 1], c='steelblue', s=100, label='Class A', edgecolors='k')
ax.scatter(class_b[:, 0], class_b[:, 1], c='salmon', s=100, label='Class B', edgecolors='k')

# New query point
query = np.array([2.5, 2.5])
ax.scatter(*query, c='gold', s=200, marker='*', label='Query point', edgecolors='k', zorder=5)

# Draw lines to 3 nearest neighbors
all_pts = np.vstack([class_a, class_b])
dists = np.linalg.norm(all_pts - query, axis=1)
nearest_idx = np.argsort(dists)[:3]
for idx in nearest_idx:
    ax.plot([query[0], all_pts[idx, 0]], [query[1], all_pts[idx, 1]],
            'k--', alpha=0.4)

ax.set_title('KNN Intuition: 3-Nearest Neighbors Vote')
ax.legend()
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

The query point is classified by whichever class has the majority among the K=3 nearest neighbors.

---
## 2. Distance Metrics

KNN relies on a notion of *distance* to decide which training points are closest. The three most common metrics are:

| Metric | Formula | Notes |
|--------|---------|-------|
| **Euclidean** (L2) | $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$ | Default; straight-line distance |
| **Manhattan** (L1) | $d = \sum_{i=1}^{n} |x_i - y_i|$ | Grid-like distance; more robust to outliers |
| **Minkowski** (Lp) | $d = \left(\sum_{i=1}^{n} |x_i - y_i|^p\right)^{1/p}$ | Generalization; p=1 is Manhattan, p=2 is Euclidean |

In [ ]:
# Computing distances manually
a = np.array([1, 2])
b = np.array([4, 6])

euclidean = np.sqrt(np.sum((a - b) ** 2))
manhattan = np.sum(np.abs(a - b))
minkowski_p3 = np.sum(np.abs(a - b) ** 3) ** (1 / 3)

print(f"Point A: {a}")
print(f"Point B: {b}")
print(f"Euclidean distance (L2): {euclidean:.4f}")
print(f"Manhattan distance (L1): {manhattan:.4f}")
print(f"Minkowski distance (p=3): {minkowski_p3:.4f}")

In [ ]:
# Visualize unit circles for different metrics to build intuition
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

theta = np.linspace(0, 2 * np.pi, 500)
metrics = [
    ('Manhattan (L1)', 1),
    ('Euclidean (L2)', 2),
    ('Minkowski (p=5)', 5),
]

for ax, (name, p) in zip(axes, metrics):
    # Points at distance 1 from origin under Lp norm
    x_vals = np.cos(theta)
    y_vals = np.sin(theta)
    # Normalize to lie on the Lp unit circle
    norms = (np.abs(x_vals) ** p + np.abs(y_vals) ** p) ** (1 / p)
    x_unit = x_vals / norms
    y_unit = y_vals / norms
    ax.plot(x_unit, y_unit, linewidth=2)
    ax.set_title(f'{name} Unit Circle')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

The shape of the unit circle shows which points are equidistant under each metric. Manhattan produces a diamond, Euclidean a circle, and higher-p Minkowski metrics approach a square.

---
## 3. How K Affects the Decision Boundary

The hyperparameter **K** controls the complexity of the model:

- **Small K** (e.g., 1): The boundary is highly sensitive to individual points, resulting in a complex, jagged surface (high variance, low bias).
- **Large K** (e.g., 50): The boundary becomes smoother and more regularized (low variance, high bias).

Let's generate a synthetic dataset and visualize decision boundaries for several values of K.

In [ ]:
# Generate synthetic 2D data
X, y = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
def plot_decision_boundary(X, y, model, ax, title):
    """Plot the decision boundary of a fitted classifier on a 2D dataset."""
    cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA'])
    cmap_bold = ListedColormap(['steelblue', 'salmon'])

    h = 0.05  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
               edgecolors='k', s=20, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

In [ ]:
# Decision boundaries for different K values
k_values = [1, 5, 20, 50]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    plot_decision_boundary(X, y, knn, ax, f'K = {k}')

plt.suptitle('Effect of K on the Decision Boundary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- K=1 produces an irregular boundary that overfits noise in the training data.
- K=5 and K=20 produce increasingly smooth boundaries.
- K=50 produces a very smooth boundary that may underfit if the true boundary is complex.

---
## 4. KNeighborsClassifier in Scikit-Learn

Scikit-learn's `KNeighborsClassifier` provides a clean API with sensible defaults:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_neighbors` | 5 | Number of neighbors (K) |
| `weights` | `'uniform'` | `'uniform'` or `'distance'` |
| `metric` | `'minkowski'` | Distance metric |
| `p` | 2 | Power parameter for Minkowski (2 = Euclidean) |

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit a basic KNN classifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Changing the distance metric
for metric_name, p_val in [('Euclidean', 2), ('Manhattan', 1)]:
    knn_m = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=p_val)
    knn_m.fit(X_train, y_train)
    acc = knn_m.score(X_test, y_test)
    print(f"{metric_name} (p={p_val}): Accuracy = {acc:.4f}")

---
## 5. The Curse of Dimensionality

As the number of features grows, the volume of the feature space increases exponentially. This has two consequences for KNN:

1. **Distances become meaningless.** In high dimensions, the ratio of the distance to the nearest neighbor versus the farthest neighbor approaches 1. All points appear roughly equidistant.
2. **Data becomes sparse.** To maintain the same density of training points, the number of samples needed grows exponentially with the number of dimensions.

In practice, KNN works best when the number of informative features is relatively small and the dataset is large enough to provide adequate coverage of the feature space.

In [ ]:
# Demonstrate: distance ratio convergence in high dimensions
dims = [2, 5, 10, 50, 100, 500, 1000]
ratios = []

for d in dims:
    # Generate 200 random points in d dimensions
    points = np.random.randn(200, d)
    # Compute distances from the origin to all points
    dists = np.linalg.norm(points, axis=1)
    ratio = dists.min() / dists.max()
    ratios.append(ratio)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, ratios, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Dimensions')
ax.set_ylabel('Min Distance / Max Distance')
ax.set_title('Curse of Dimensionality: Distance Ratio Approaches 1')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("As dimensions grow, the nearest and farthest points become indistinguishable.")

---
## 6. Why Feature Scaling Is Critical for KNN

Because KNN uses distance to find neighbors, features with larger numeric ranges will dominate the distance calculation. For example, if feature A ranges from 0 to 1000 and feature B ranges from 0 to 1, differences in A will overwhelm differences in B.

**Solution:** Always scale features before applying KNN. Common approaches are standardization (zero mean, unit variance) and min-max normalization.

In [ ]:
# Create data where one feature has a much larger range
X_raw, y_raw = make_classification(
    n_samples=400, n_features=2, n_informative=2, n_redundant=0,
    random_state=7
)

# Artificially scale feature 0 by 1000
X_unscaled = X_raw.copy()
X_unscaled[:, 0] = X_unscaled[:, 0] * 1000

print(f"Feature 0 range: [{X_unscaled[:, 0].min():.1f}, {X_unscaled[:, 0].max():.1f}]")
print(f"Feature 1 range: [{X_unscaled[:, 1].min():.4f}, {X_unscaled[:, 1].max():.4f}]")

In [ ]:
# Split
X_tr_un, X_te_un, y_tr_un, y_te_un = train_test_split(
    X_unscaled, y_raw, test_size=0.25, random_state=42
)

# Without scaling
knn_noscale = KNeighborsClassifier(n_neighbors=5)
knn_noscale.fit(X_tr_un, y_tr_un)
acc_noscale = knn_noscale.score(X_te_un, y_te_un)

# With scaling
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_un)
X_te_sc = scaler.transform(X_te_un)

knn_scale = KNeighborsClassifier(n_neighbors=5)
knn_scale.fit(X_tr_sc, y_tr_un)
acc_scale = knn_scale.score(X_te_sc, y_te_un)

print(f"Accuracy WITHOUT scaling: {acc_noscale:.4f}")
print(f"Accuracy WITH scaling:    {acc_scale:.4f}")

In [ ]:
# Visualize the decision boundaries side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary(X_tr_un, y_tr_un, knn_noscale, axes[0],
                       f'Without Scaling (acc={acc_noscale:.2f})')
plot_decision_boundary(X_tr_sc, y_tr_un, knn_scale, axes[1],
                       f'With Scaling (acc={acc_scale:.2f})')

plt.suptitle('Impact of Feature Scaling on KNN', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Without scaling, KNN effectively ignores the smaller feature and draws near-vertical boundaries based only on the large-range feature.

---
## 7. Classification Example on Synthetic Data

Let's use the `make_moons` dataset, which creates two interleaving half circles — a non-linearly separable problem where KNN excels.

In [ ]:
# Generate moons dataset
X_moons, y_moons = make_moons(n_samples=500, noise=0.25, random_state=42)

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=42
)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons,
                     cmap=ListedColormap(['steelblue', 'salmon']),
                     edgecolors='k', s=30, alpha=0.7)
ax.set_title('Make Moons Dataset')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

In [ ]:
# Scale features and fit KNN using a Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=7))
])

pipe.fit(X_m_train, y_m_train)
y_m_pred = pipe.predict(X_m_test)

print(f"Accuracy on moons test set: {accuracy_score(y_m_test, y_m_pred):.4f}")
print()
print(classification_report(y_m_test, y_m_pred))

---
## 8. Visualizing Decision Boundaries for Different K Values

Using the moons dataset, we can see how different K values capture the non-linear boundary.

In [ ]:
# Scale the moons data for visualization
scaler_m = StandardScaler()
X_m_scaled = scaler_m.fit_transform(X_moons)

k_values_moons = [1, 5, 15, 50]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, k in zip(axes, k_values_moons):
    knn_m = KNeighborsClassifier(n_neighbors=k)
    knn_m.fit(X_m_scaled, y_moons)
    acc = knn_m.score(X_m_scaled, y_moons)
    plot_decision_boundary(X_m_scaled, y_moons, knn_m, ax,
                           f'K={k} (train acc={acc:.2f})')

plt.suptitle('Decision Boundaries on Moons Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

K=1 perfectly memorizes the training data but is noisy. Moderate values like K=5 or K=15 capture the crescent shapes cleanly. K=50 oversmooths and begins to misclassify points near the boundary.

---
## 9. Choosing K: The Elbow Method

A common approach is to train KNN for a range of K values and plot accuracy (or error rate) against K. The point where accuracy levels off or starts to decrease is a good choice.

In [ ]:
# Elbow method: plot K vs test accuracy
k_range = range(1, 51)
train_accs = []
test_accs = []

# Use the scaled moons data
X_m_tr_sc = scaler_m.transform(X_m_train)
X_m_te_sc = scaler_m.transform(X_m_test)

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_m_tr_sc, y_m_train)
    train_accs.append(knn_k.score(X_m_tr_sc, y_m_train))
    test_accs.append(knn_k.score(X_m_te_sc, y_m_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_accs, 'o-', label='Train Accuracy', alpha=0.7)
ax.plot(k_range, test_accs, 's-', label='Test Accuracy', alpha=0.7)
ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('Elbow Method: Choosing K')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(test_accs)]
print(f"Best K on test set: {best_k} (accuracy = {max(test_accs):.4f})")

Note that K=1 always achieves perfect training accuracy (each point is its own nearest neighbor) but may perform poorly on unseen data.

---
## 10. Cross-Validation for K Selection

The elbow method on a single train/test split can be noisy. Cross-validation provides a more robust estimate by averaging performance across multiple folds.

In [ ]:
# 5-fold cross-validation for each K
k_range_cv = range(1, 41)
cv_means = []
cv_stds = []

for k in k_range_cv:
    pipe_cv = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe_cv, X_moons, y_moons, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(list(k_range_cv), cv_means, 'o-', color='steelblue', label='Mean CV Accuracy')
ax.fill_between(list(k_range_cv),
                cv_means - cv_stds,
                cv_means + cv_stds,
                alpha=0.2, color='steelblue', label='\u00b11 Std Dev')

best_k_cv = list(k_range_cv)[np.argmax(cv_means)]
ax.axvline(best_k_cv, color='salmon', linestyle='--', label=f'Best K = {best_k_cv}')

ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Cross-Validation for K Selection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best K via cross-validation: {best_k_cv}")
print(f"Mean CV accuracy: {cv_means.max():.4f} (+/- {cv_stds[np.argmax(cv_means)]:.4f})")

The shaded region shows the standard deviation across folds. The best K balances accuracy and stability.

---
## 11. Pros and Cons of KNN

### Advantages

- **Simple and intuitive** — easy to understand and implement.
- **No training phase** — fitting is instant (just store the data).
- **Naturally handles multi-class** problems without modification.
- **Non-parametric** — makes no assumptions about the data distribution, so it can capture complex, non-linear decision boundaries.

### Disadvantages

- **Slow prediction** — computing distances to all training points is O(n * d) per query, which becomes expensive for large datasets.
- **Memory intensive** — the entire training set must be stored.
- **Sensitive to irrelevant features** — noisy or uninformative features distort distances.
- **Curse of dimensionality** — performance degrades in high-dimensional spaces.
- **Requires feature scaling** — results are unreliable if features are on different scales.

---
## Summary

In this notebook we covered:

1. **KNN intuition** — classify by majority vote of the K nearest training points.
2. **Distance metrics** — Euclidean, Manhattan, and Minkowski measure closeness differently.
3. **Effect of K** — small K overfits, large K underfits; K controls the bias-variance trade-off.
4. **Scikit-learn API** — `KNeighborsClassifier` with configurable neighbors, weights, and metrics.
5. **Curse of dimensionality** — distances lose meaning in high dimensions.
6. **Feature scaling** — essential for any distance-based method.
7. **K selection** — use the elbow method or cross-validation to find the best K.